<a href="https://colab.research.google.com/github/ronik18/earnalism-digital-library/blob/main/colab/Earnalism_Audiobook_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Earnalism Autonomous Audiobook Pipeline

This notebook creates a private, evidence-bound audiobook candidate from an approved Earnalism controlled-publication artifact.

**Default production candidate:** *The Art of Money Getting* by P. T. Barnum  
**Voice:** Kokoro's bm_george with bounded, sentence-aware pacing  
**Release behavior:** fail closed. It does not publish, expose audio, or write storage credentials.

The decision engine may continue, perform one targeted repair, or stop. It never invents listening scores and cannot mark public release ready until rights, manuscript integrity, measured timing, ASR, full-title listening, storage, endpoint, and browser evidence all pass.


In [ ]:
#@title 1. Production parameters
BOOK_SLUG = "the-art-of-money-getting" #@param {type:"string"}
VOICE = "bm_george" #@param {type:"string"}
MODEL_REPO = "hexgrad/Kokoro-82M" #@param {type:"string"}
REPO_REF = "main" #@param {type:"string"}
BASE_SPEED = 0.92 #@param {type:"slider", min:0.86, max:0.98, step:0.01}
PERSIST_TO_GOOGLE_DRIVE = False #@param {type:"boolean"}
GO_LIVE_ENABLED = False #@param {type:"boolean"}
RELEASE_ENDPOINT_SECRET = "EARNALISM_B2_PRESIGN_ENDPOINT"
ADMIN_TOKEN_SECRET = "EARNALISM_B2_ADMIN_TOKEN"
B2_AUDIO_OBJECT_KEY = "" #@param {type:"string"}
OWNER_FULL_GENERATION_APPROVED = True #@param {type:"boolean"}
OWNER_PUBLIC_RELEASE_INTENT = True #@param {type:"boolean"}

ASR_MODEL = "large-v3-turbo"
ASR_SCORE_MIN = 0.97
COVERAGE_MIN = 0.98
LISTENING_SCORE_MIN = 8.9
LISTENING_CONFIDENCE_MIN = 0.90
SPEED_MIN, SPEED_MAX = 0.86, 0.98
MAX_WORDS_PER_UNIT = 80
MAX_TARGETED_REPAIRS = 1
SAMPLE_RATE = 24000
MP3_BITRATE = "96k"
PUBLICATION_TERRITORY = "IN"

assert BOOK_SLUG and VOICE
assert SPEED_MIN <= BASE_SPEED <= SPEED_MAX
assert OWNER_FULL_GENERATION_APPROVED, "Full generation requires explicit owner approval for this title and voice."
print({"slug": BOOK_SLUG, "voice": VOICE, "intent": "PRIVATE_CANDIDATE_THEN_GO_LIVE_AFTER_ALL_GATES"})


{'slug': 'the-art-of-money-getting', 'voice': 'bm_george', 'intent': 'PRIVATE_CANDIDATE_THEN_GO_LIVE_AFTER_ALL_GATES'}


## 2. Runtime and durable workspace

Use a T4 or L4 GPU. The notebook stores resumable unit audio and evidence under Google Drive when enabled.


In [ ]:
import os, re, sys, json, math, time, hashlib, base64, shutil, subprocess
from pathlib import Path
from datetime import datetime, timezone

subprocess.run(["apt-get", "-qq", "update"], check=True)
subprocess.run(["apt-get", "-qq", "install", "-y", "ffmpeg", "espeak-ng"], check=True)
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "kokoro", "soundfile", "faster-whisper", "rapidfuzz", "requests"
], check=True)

import numpy as np
import soundfile as sf
import torch
import requests
from rapidfuzz.fuzz import ratio

assert torch.cuda.is_available(), "Switch Colab to a T4/L4 GPU runtime before continuing."

OUTPUT_ROOT = Path("/content/earnalism-audiobooks")

def colab_secret(name):
    value = os.environ.get(name, "")
    if value:
        return value
    try:
        from google.colab import userdata
        return userdata.get(name) or ""
    except Exception:
        return ""

B2_AUDIO_UPLOAD_URL = ""
RELEASE_ENDPOINT = colab_secret(RELEASE_ENDPOINT_SECRET).rstrip("/")
ADMIN_TOKEN = colab_secret(ADMIN_TOKEN_SECRET)
if GO_LIVE_ENABLED:
    assert RELEASE_ENDPOINT and ADMIN_TOKEN, "GO_LIVE_ENABLED requires the Railway release endpoint and admin token in Colab Secrets."
    assert B2_AUDIO_OBJECT_KEY, "GO_LIVE_ENABLED requires the production MP3 object key."
    presign_response = requests.post(f"{RELEASE_ENDPOINT}/api/admin/books/{BOOK_SLUG}/audiobook/presign", headers={"Authorization": f"Bearer {ADMIN_TOKEN}"}, json={"audio_object_key": B2_AUDIO_OBJECT_KEY}, timeout=30)
    presign_response.raise_for_status()
    B2_AUDIO_UPLOAD_URL = presign_response.json()["objects"]["audio"]["url"]
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

REPO = Path("/content/earnalism-digital-library")
if not REPO.exists():
    subprocess.run([
        "git", "clone", "--depth", "1", "--branch", REPO_REF,
        "https://github.com/ronik18/earnalism-digital-library.git", str(REPO)
    ], check=True)
else:
    subprocess.run(["git", "-C", str(REPO), "fetch", "origin", REPO_REF, "--depth", "1"], check=True)
    subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", "FETCH_HEAD"], check=True)

RUN = OUTPUT_ROOT / f"{BOOK_SLUG}-kokoro-{VOICE}"
UNITS_DIR = RUN / "units"
SAMPLES_DIR = RUN / "listening_samples"
ASR_DIR = RUN / "asr_chunks"
for folder in (RUN, UNITS_DIR, SAMPLES_DIR, ASR_DIR):
    folder.mkdir(parents=True, exist_ok=True)

print({"torch": torch.__version__, "cuda": True, "run": str(RUN), "repo_ref": REPO_REF})


{'torch': '2.11.0+cu128', 'cuda': True, 'run': '/content/earnalism-audiobooks/the-art-of-money-getting-kokoro-bm_george', 'repo_ref': 'main'}


## 3. Resolve and validate canonical publication truth

No manuscript upload is accepted. The notebook reads only the selected controlled-publication artifact from the repository and verifies its reader, rights, chapter, cover, and checksum evidence.


In [ ]:
def load_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))

def sha256_bytes(value):
    return hashlib.sha256(value).hexdigest()

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def sha256_text(value):
    return sha256_bytes(value.encode("utf-8"))

def collapse_ws(value):
    return re.sub(r"\s+", " ", value or "").strip()

def now_iso():
    return datetime.now(timezone.utc).isoformat()

ARTIFACT = REPO / "data" / "controlled_publications" / BOOK_SLUG
required = [
    "public_book.json", "reader_manifest.json", "approval_evidence.json",
    "source_evidence.json", "checksum_manifest.json"
]
missing = [name for name in required if not (ARTIFACT / name).is_file()]
assert not missing, f"Missing controlled artifacts: {missing}"

book = load_json(ARTIFACT / "public_book.json")
reader = load_json(ARTIFACT / "reader_manifest.json")
approval = load_json(ARTIFACT / "approval_evidence.json")
source = load_json(ARTIFACT / "source_evidence.json")
checksums = load_json(ARTIFACT / "checksum_manifest.json")

assert book.get("slug") == BOOK_SLUG
assert book.get("publication_status") == "LIVE_APPROVED"
assert book.get("is_published") is True
assert book.get("qa_status") == "QA_PASSED"
assert approval.get("approved_to_publish") is True
assert approval.get("rights_tier") == "A"
assert source.get("reader_facing_boilerplate_removed") is True
assert source.get("source_url") and source.get("rights_basis")
assert book.get("audio_enabled") is False and book.get("audiobook_enabled") is False
assert book.get("cover_image_url") or book.get("cover_url"), "Approved cover URL is required."

checksum_failures = []
for row in checksums.get("files", []):
    rel = row.get("file")
    if not rel or rel == "checksum_manifest.json":
        continue
    target = ARTIFACT / rel
    if not target.is_file() or sha256_file(target) != row.get("sha256"):
        checksum_failures.append(rel)
assert not checksum_failures, f"Controlled artifact checksum mismatch: {checksum_failures}"

excluded_titles = re.compile(r"^(contents?|index|bibliography|references?|front cover|back cover)$", re.I)
all_chapter_meta = sorted(book.get("chapters") or [], key=lambda row: int(row.get("order") or 0))
assert [int(row.get("order") or 0) for row in all_chapter_meta] == list(range(1, len(all_chapter_meta) + 1)), "Controlled chapter index is not contiguous."
chapters = []
for meta in all_chapter_meta:
    assert meta.get("processing_status") == "ready", f"Chapter not ready: {meta.get('id')}"
    if excluded_titles.match(str(meta.get("title") or "").strip()):
        continue
    chapter_path = ARTIFACT / "chapters" / f"{meta['id']}.json"
    payload = load_json(chapter_path)
    content = str(payload.get("content") or "")
    assert content.strip(), f"Empty chapter: {meta['id']}"
    assert payload.get("content_hash") == sha256_text(content), f"Chapter content hash mismatch: {meta['id']}"
    chapters.append({**meta, "content": content, "file_sha256": sha256_file(chapter_path)})
assert chapters, "No narratable chapters remain after exclusions."

canonical_source = "\n\n".join(row["content"].strip() for row in chapters).strip() + "\n"
forbidden = re.compile(r"project gutenberg|gutenberg\.org|license agreement|terms of use|distributed proofreaders", re.I)
assert not forbidden.search(canonical_source), "Reader-facing source boilerplate remains in the manuscript."

# Whitespace normalization repairs historical hard-wrapped source lines without changing token order or punctuation.
narrated_source = collapse_ws(canonical_source) + "\n"
CANONICAL_SOURCE_SHA256 = sha256_text(canonical_source)
MANUSCRIPT_SHA256 = sha256_text(narrated_source)
SOURCE_PATH = RUN / f"{BOOK_SLUG}_narrated_manuscript.txt"
SOURCE_PATH.write_text(narrated_source, encoding="utf-8")

input_manifest = {
    "schema_version": "earnalism.kokoro_colab_input.v1",
    "slug": BOOK_SLUG,
    "title": book.get("title"),
    "author": book.get("author"),
    "language": "en",
    "controlled_artifact_path": f"data/controlled_publications/{BOOK_SLUG}",
    "canonical_source_sha256": CANONICAL_SOURCE_SHA256,
    "sanitized_source_sha256": MANUSCRIPT_SHA256,
    "text_equivalence_mode": "collapse_whitespace_only",
    "chapter_count": len(chapters),
    "chapter_orders": [row["order"] for row in chapters],
    "chapter_file_sha256": {row["id"]: row["file_sha256"] for row in chapters},
    "rights_status": "PASS",
    "reader_status": "PASS",
    "cover_status": "PASS",
    "audio_initially_hidden": True,
    "publication_territory": PUBLICATION_TERRITORY,
    "generated_at": now_iso(),
}
INPUT_MANIFEST_PATH = RUN / "input_manifest.json"
INPUT_MANIFEST_PATH.write_text(json.dumps(input_manifest, indent=2, ensure_ascii=False), encoding="utf-8")
print({"title": book.get("title"), "words": len(narrated_source.split()), "chapters": len(chapters), "manuscript_sha256": MANUSCRIPT_SHA256})


{'title': 'The Art of Money Getting', 'words': 13550, 'chapters': 1, 'manuscript_sha256': 'c360a8193e596c8ee1b20f49449ba528a3f8bb658c71590971cc394e6090048d'}


## 4. Build a punctuation-aware prosody plan

Punctuation is preserved. The decision engine selects a bounded speaking rate for each sentence and inserts measured inter-sentence and inter-paragraph pauses. It does not rewrite the author's words.


In [ ]:
ABBREVIATION_RE = re.compile(r"(?:\b(?:Mr|Mrs|Ms|Dr|St|Prof|Sr|Jr|vs|etc)|\b[A-Z])\.$", re.I)

def split_sentences(text):
    raw_parts = re.split(r"(?<=[.!?])\s+(?=[\"'A-Z0-9])", collapse_ws(text))
    merged = []
    for part in raw_parts:
        if merged and ABBREVIATION_RE.search(merged[-1].strip()):
            merged[-1] += " " + part
        else:
            merged.append(part)
    final = []
    for sentence in merged:
        if len(sentence.split()) <= MAX_WORDS_PER_UNIT:
            final.append(sentence.strip())
            continue
        pieces = re.split(r"(?<=[;:])\s+", sentence)
        buffer = ""
        for piece in pieces:
            candidate = (buffer + " " + piece).strip()
            if buffer and len(candidate.split()) > MAX_WORDS_PER_UNIT:
                final.append(buffer)
                buffer = piece.strip()
            else:
                buffer = candidate
        if buffer:
            final.append(buffer)
    return [row for row in final if row]

def choose_speed(text):
    words = len(text.split())
    speed = BASE_SPEED
    if words >= 42: speed -= 0.035
    elif words >= 30: speed -= 0.02
    elif words <= 8: speed += 0.015
    if re.search(r"[;:\\u2014]", text): speed -= 0.015
    if "?" in text: speed -= 0.02
    if "!" in text: speed -= 0.025
    if text.count(",") >= 3: speed -= 0.015
    return round(min(SPEED_MAX, max(SPEED_MIN, speed)), 3)

def choose_pause_ms(text, paragraph_end):
    if paragraph_end: return 620
    if text.rstrip().endswith("!"): return 360
    if text.rstrip().endswith("?"): return 330
    if text.rstrip().endswith((":", ";")): return 250
    return 190

sentence_rows = []
for chapter in chapters:
    for sentence in split_sentences(chapter["content"]):
        sentence_rows.append({"chapter_id": chapter["id"], "text": sentence})
sentences = [row["text"] for row in sentence_rows]
assert len(sentences) >= 20, "Manuscript segmentation produced too few units."
assert collapse_ws(" ".join(sentences)) == collapse_ws(narrated_source), "Segmentation changed manuscript text."

units = []
paragraph_id = 1
paragraph_words = 0
for index, sentence_row in enumerate(sentence_rows):
    sentence = sentence_row["text"]
    word_count = len(sentence.split())
    paragraph_words += word_count
    is_last = index == len(sentences) - 1
    paragraph_end = is_last or paragraph_words >= 105 or (paragraph_words >= 55 and sentence.rstrip().endswith(("?", "!", ":")))
    unit = {
        "unit_id": f"unit-{index + 1:05d}",
        "order": index + 1,
        "chapter_id": sentence_row["chapter_id"],
        "paragraph_id": f"paragraph-{paragraph_id:05d}",
        "text": sentence,
        "text_sha256": sha256_text(sentence),
        "word_count": word_count,
        "speed": choose_speed(sentence),
        "pause_ms": choose_pause_ms(sentence, paragraph_end),
        "paragraph_end": paragraph_end,
    }
    units.append(unit)
    if paragraph_end:
        paragraph_id += 1
        paragraph_words = 0

plan = {
    "schema_version": "earnalism.kokoro_measured_unit_plan.v1",
    "slug": BOOK_SLUG,
    "voice": VOICE,
    "model": MODEL_REPO,
    "base_speed": BASE_SPEED,
    "speed_bounds": [SPEED_MIN, SPEED_MAX],
    "source_sha256": MANUSCRIPT_SHA256,
    "canonical_source_sha256": CANONICAL_SOURCE_SHA256,
    "unit_count": len(units),
    "paragraph_count": len({row["paragraph_id"] for row in units}),
    "units": units,
}
PLAN_PATH = RUN / "generation_plan.json"
PLAN_PATH.write_text(json.dumps(plan, indent=2, ensure_ascii=False), encoding="utf-8")
print({"units": len(units), "paragraphs": plan["paragraph_count"], "speed_range": [min(u["speed"] for u in units), max(u["speed"] for u in units)]})


{'units': 456, 'paragraphs': 108, 'speed_range': [0.86, 0.935]}


## 5. Initialize Kokoro and synthesis helpers

Every unit is hash-bound and resumable. Existing audio is reused only when its text, model, voice, speed, and attempt fingerprint are unchanged.


In [1]:
from kokoro import KPipeline
from faster_whisper import WhisperModel


pipeline = KPipeline(lang_code="a", repo_id=MODEL_REPO, device="cuda")

def audio_from_result(result):
    if hasattr(result, "output") and hasattr(result.output, "audio"):
        return result.output.audio.detach().cpu().numpy()
    if isinstance(result, (tuple, list)) and len(result) >= 3:
        audio = result[2]
        return audio.detach().cpu().numpy() if hasattr(audio, "detach") else np.asarray(audio)
    raise RuntimeError("Unsupported Kokoro result shape")

def synth_text(text, speed, output_path):
    pieces = []
    for result in pipeline(text, voice=VOICE, speed=float(speed), split_pattern=r"(?<=[.!?])\s+"):
        pieces.append(audio_from_result(result).astype(np.float32))
    assert pieces, "Kokoro returned no audio."
    audio = np.concatenate(pieces)
    assert np.isfinite(audio).all() and len(audio) > SAMPLE_RATE // 2
    peak = float(np.max(np.abs(audio)))
    if peak > 0.985:
        audio = audio * (0.975 / peak)
    sf.write(output_path, audio, SAMPLE_RATE, subtype="PCM_16")
    return {
        "duration_seconds": len(audio) / SAMPLE_RATE,
        "peak": float(np.max(np.abs(audio))),
        "rms": float(np.sqrt(np.mean(np.square(audio)))),
        "clipping_ratio": float(np.mean(np.abs(audio) >= 0.999)),
    }

def normalized_words(text):
    return re.findall(r"[a-z0-9]+", text.lower().replace("'", "'"))

def compare_transcript(source_text, transcript):
    source_words = normalized_words(source_text)
    asr_words = normalized_words(transcript)
    similarity = ratio(" ".join(source_words), " ".join(asr_words)) / 100
    coverage = min(len(source_words), len(asr_words)) / max(1, len(source_words))
    first = ratio(" ".join(source_words[:20]), " ".join(asr_words[:20])) / 100
    last = ratio(" ".join(source_words[-20:]), " ".join(asr_words[-20:])) / 100
    return {"similarity": similarity, "coverage": coverage, "first_span_score": first, "last_span_score": last, "source_words": len(source_words), "asr_words": len(asr_words)}

ATTEMPT_FINGERPRINT = sha256_text(json.dumps({
    "slug": BOOK_SLUG, "source_sha256": MANUSCRIPT_SHA256, "model": MODEL_REPO,
    "voice": VOICE, "base_speed": BASE_SPEED,
    "unit_hashes": [row["text_sha256"] for row in units]
}, sort_keys=True, separators=(",", ":")))

fingerprint_path = RUN / "attempt_fingerprint.txt"
if fingerprint_path.exists():
    assert fingerprint_path.read_text().strip() == ATTEMPT_FINGERPRINT, "Run directory belongs to a different immutable attempt."
else:
    fingerprint_path.write_text(ATTEMPT_FINGERPRINT)
print({"attempt_fingerprint": ATTEMPT_FINGERPRINT, "model": MODEL_REPO, "voice": VOICE})


ModuleNotFoundError: No module named 'kokoro'

## 6. Representative objective audition and one bounded repair

Six manuscript-bound samples cover opening, ending, dense prose, punctuation, questions, and the middle. Failed samples receive at most one slower targeted repair. The notebook stops before full generation if objective fidelity remains below policy.


In [ ]:
def representative_blocks():
    scores = {
        "opening": 0,
        "ending": max(0, len(units) - 4),
        "middle": max(0, len(units) // 2 - 2),
        "punctuation": max(range(len(units)), key=lambda i: sum(units[i]["text"].count(mark) for mark in ",;:\\u2014!?")),
        "question": next((i for i,u in enumerate(units) if "?" in u["text"]), len(units)//3),
        "dense": max(range(len(units)), key=lambda i: units[i]["word_count"]),
    }
    result = {}
    for label, anchor in scores.items():
        start = min(max(0, anchor), max(0, len(units) - 4))
        result[label] = units[start:start+4]
    return result

rep_asr = WhisperModel("small.en", device="cuda", compute_type="float16")
representative_results = []
for label, block in representative_blocks().items():
    text_value = " ".join(row["text"] for row in block)
    speed_value = round(sum(row["speed"] for row in block) / len(block), 3)
    wav_path = SAMPLES_DIR / f"representative_{label}.wav"
    metrics = synth_text(text_value, speed_value, wav_path)
    segments, info = rep_asr.transcribe(str(wav_path), language="en", beam_size=5, vad_filter=True)
    transcript = " ".join(segment.text for segment in segments).strip()
    comparison = compare_transcript(text_value, transcript)
    passed = comparison["similarity"] >= ASR_SCORE_MIN and comparison["coverage"] >= COVERAGE_MIN and comparison["first_span_score"] >= 0.90 and comparison["last_span_score"] >= 0.90 and metrics["clipping_ratio"] <= 0.0001
    repairs = 0
    if not passed and MAX_TARGETED_REPAIRS:
        repairs = 1
        metrics = synth_text(text_value, max(SPEED_MIN, speed_value - 0.02), wav_path)
        segments, info = rep_asr.transcribe(str(wav_path), language="en", beam_size=5, vad_filter=True)
        transcript = " ".join(segment.text for segment in segments).strip()
        comparison = compare_transcript(text_value, transcript)
        passed = comparison["similarity"] >= ASR_SCORE_MIN and comparison["coverage"] >= COVERAGE_MIN and comparison["first_span_score"] >= 0.90 and comparison["last_span_score"] >= 0.90 and metrics["clipping_ratio"] <= 0.0001
    representative_results.append({"label": label, "unit_ids": [row["unit_id"] for row in block], "text_sha256": sha256_text(text_value), "audio_sha256": sha256_file(wav_path), "metrics": metrics, "asr": comparison, "repairs": repairs, "passed": passed, "file": wav_path.name})

representative_gate = all(row["passed"] for row in representative_results)
REPRESENTATIVE_QA_PATH = RUN / "representative_objective_qa.json"
REPRESENTATIVE_QA_PATH.write_text(json.dumps({"schema_version":"earnalism.kokoro_representative_objective_qa.v1","slug":BOOK_SLUG,"attempt_fingerprint":ATTEMPT_FINGERPRINT,"thresholds":{"asr":ASR_SCORE_MIN,"coverage":COVERAGE_MIN},"samples":representative_results,"passed":representative_gate}, indent=2), encoding="utf-8")
print(json.dumps({"decision":"PROCEED_FULL_GENERATION" if representative_gate else "TERMINAL_BLOCKED_WITH_EVIDENCE","samples":[{"label":r["label"],"asr":round(r["asr"]["similarity"],4),"coverage":round(r["asr"]["coverage"],4),"repairs":r["repairs"],"passed":r["passed"]} for r in representative_results]}, indent=2))
assert representative_gate, "Representative objective gate failed after the only allowed targeted repair."


voices/bm_george.pt: reconstructing file:   0%|          |  0.00B /  523kB            

voices/bm_george.pt: downloading bytes:           |  0.00B            

{
  "decision": "TERMINAL_BLOCKED_WITH_EVIDENCE",
  "samples": [
    {
      "label": "opening",
      "asr": 1.0,
      "coverage": 1.0,
      "repairs": 0,
      "passed": true
    },
    {
      "label": "ending",
      "asr": 1.0,
      "coverage": 1.0,
      "repairs": 0,
      "passed": true
    },
    {
      "label": "middle",
      "asr": 1.0,
      "coverage": 1.0,
      "repairs": 0,
      "passed": true
    },
    {
      "label": "punctuation",
      "asr": 0.9988,
      "coverage": 1.0,
      "repairs": 0,
      "passed": true
    },
    {
      "label": "question",
      "asr": 0.9939,
      "coverage": 0.9878,
      "repairs": 0,
      "passed": true
    },
    {
      "label": "dense",
      "asr": 0.9667,
      "coverage": 0.9627,
      "repairs": 1,
      "passed": false
    }
  ]
}


AssertionError: Representative objective gate failed after the only allowed targeted repair.

## 7. Resumable full-title generation

This is the single approved private full-title generation attempt. Restarting the cell resumes identical units; it does not create another attempt fingerprint.


In [ ]:
records = []
for index, unit in enumerate(units):
    wav_path = UNITS_DIR / f"{unit['unit_id']}.wav"
    sidecar = UNITS_DIR / f"{unit['unit_id']}.json"
    expected = {"attempt_fingerprint":ATTEMPT_FINGERPRINT,"unit_id":unit["unit_id"],"text_sha256":unit["text_sha256"],"voice":VOICE,"model":MODEL_REPO,"speed":unit["speed"]}
    reusable = False
    if wav_path.exists() and sidecar.exists():
        prior = load_json(sidecar)
        reusable = all(prior.get(key) == value for key,value in expected.items()) and prior.get("audio_sha256") == sha256_file(wav_path) and wav_path.stat().st_size > 1000
    if not reusable:
        metrics = synth_text(unit["text"], unit["speed"], wav_path)
        record = {**expected, **metrics, "audio_sha256":sha256_file(wav_path), "size_bytes":wav_path.stat().st_size, "pause_ms":unit["pause_ms"], "paragraph_id":unit["paragraph_id"], "chapter_id":unit["chapter_id"]}
        sidecar.write_text(json.dumps(record, indent=2), encoding="utf-8")
    else:
        record = prior
    records.append(record)
    if (index + 1) % 25 == 0 or index + 1 == len(units):
        checkpoint = {"schema_version":"earnalism.kokoro_full_generation_checkpoint.v1","slug":BOOK_SLUG,"attempt_fingerprint":ATTEMPT_FINGERPRINT,"completed":index+1,"total":len(units),"updated_at":now_iso(),"records":records}
        (RUN / "generation_checkpoint.json").write_text(json.dumps(checkpoint, indent=2), encoding="utf-8")
        print(f"{index+1}/{len(units)} units complete", flush=True)
print("FULL_SYNTHESIS_COMPLETE")


## 8. Assemble delivery audio and frame-measured synchronization

Because every sentence is rendered separately, the notebook measures each WAV's exact frame count before concatenation. Sentence and paragraph boundaries therefore come from audio frames, not estimated speech rates.


In [ ]:
MASTER_WAV = RUN / f"{BOOK_SLUG}_{VOICE}_master.wav"
FINAL_MP3 = RUN / f"{BOOK_SLUG}_{VOICE}.mp3"
timing_rows = []
current_frame = 0
with sf.SoundFile(MASTER_WAV, mode="w", samplerate=SAMPLE_RATE, channels=1, subtype="PCM_16") as master:
    for unit, record in zip(units, records):
        wav_path = UNITS_DIR / f"{unit['unit_id']}.wav"
        audio, sr = sf.read(wav_path, dtype="float32")
        assert sr == SAMPLE_RATE
        if audio.ndim > 1: audio = audio.mean(axis=1)
        start = current_frame / SAMPLE_RATE
        master.write(audio)
        current_frame += len(audio)
        speech_end = current_frame / SAMPLE_RATE
        pause_frames = int(round(unit["pause_ms"] * SAMPLE_RATE / 1000))
        master.write(np.zeros(pause_frames, dtype=np.float32))
        current_frame += pause_frames
        timing_rows.append({"unit_id":unit["unit_id"],"chapter_id":unit["chapter_id"],"paragraph_id":unit["paragraph_id"],"text":unit["text"],"text_sha256":unit["text_sha256"],"start":round(start,6),"end":round(speech_end,6),"pause_end":round(current_frame/SAMPLE_RATE,6),"measurement":"exact_pcm_frame_boundary"})

subprocess.run(["ffmpeg","-y","-loglevel","error","-i",str(MASTER_WAV),"-ac","1","-ar",str(SAMPLE_RATE),"-codec:a","libmp3lame","-b:a",MP3_BITRATE,str(FINAL_MP3)], check=True)
subprocess.run(["ffmpeg","-v","error","-i",str(FINAL_MP3),"-f","null","-"], check=True)
AUDIO_SHA256 = sha256_file(FINAL_MP3)
DURATION_SECONDS = float(subprocess.check_output(["ffprobe","-v","error","-show_entries","format=duration","-of","default=noprint_wrappers=1:nokey=1",str(FINAL_MP3)]).decode().strip())

paragraphs = []
for paragraph_id in dict.fromkeys(row["paragraph_id"] for row in timing_rows):
    rows = [row for row in timing_rows if row["paragraph_id"] == paragraph_id]
    paragraphs.append({"id":paragraph_id,"start":rows[0]["start"],"end":rows[-1]["end"],"text":" ".join(row["text"] for row in rows),"unit_ids":[row["unit_id"] for row in rows]})

TIMESTAMPS_PATH = RUN / f"{BOOK_SLUG}_timestamps.json"
TIMESTAMPS_PATH.write_text(json.dumps({"schema_version":"earnalism.measured_synthesis_timestamps.v1","slug":BOOK_SLUG,"audio_hash":AUDIO_SHA256,"source_text_hash":MANUSCRIPT_SHA256,"sync_granularity":"measured_paragraph","alignment_method":"exact_synthesis_pcm_frame_boundaries","auto_estimated_sync":False,"sample_rate":SAMPLE_RATE,"duration_seconds":DURATION_SECONDS,"units":timing_rows,"paragraphs":paragraphs}, indent=2, ensure_ascii=False), encoding="utf-8")

def vtt_time(seconds):
    ms = int(round(seconds * 1000)); hours, rem = divmod(ms, 3600000); minutes, rem = divmod(rem, 60000); secs, millis = divmod(rem, 1000)
    return f"{hours:02d}:{minutes:02d}:{secs:02d}.{millis:03d}"
VTT_PATH = RUN / f"{BOOK_SLUG}_highlight.vtt"
with VTT_PATH.open("w", encoding="utf-8") as handle:
    handle.write("WEBVTT\n\n")
    for index, row in enumerate(paragraphs, 1):
        handle.write(f"{index}\n{vtt_time(row['start'])} --> {vtt_time(row['end'])}\n{row['text']}\n\n")

chapter_timing = []
for chapter in chapters:
    rows = [row for row in timing_rows if row["chapter_id"] == chapter["id"]]
    assert rows, f"No generated audio mapped to chapter {chapter['id']}"
    chapter_timing.append({"id":chapter["id"],"title":chapter["title"],"order":chapter["order"],"start_seconds":rows[0]["start"],"end_seconds":rows[-1]["pause_end"]})
CHAPTERS_PATH = RUN / f"{BOOK_SLUG}_chapters.json"
CHAPTERS_PATH.write_text(json.dumps({"slug":BOOK_SLUG,"audio_hash":AUDIO_SHA256,"chapters":chapter_timing}, indent=2), encoding="utf-8")
META_PATH = RUN / f"{BOOK_SLUG}_meta.json"
META_PATH.write_text(json.dumps({"slug":BOOK_SLUG,"title":book.get("title"),"author":book.get("author"),"provider":"kokoro","model":MODEL_REPO,"voice":VOICE,"base_speed":BASE_SPEED,"audio_hash":AUDIO_SHA256,"source_text_hash":MANUSCRIPT_SHA256,"duration_seconds":DURATION_SECONDS,"sync_granularity":"measured_paragraph","alignment_method":"exact_synthesis_pcm_frame_boundaries","auto_estimated_sync":False,"attempt_fingerprint":ATTEMPT_FINGERPRINT}, indent=2), encoding="utf-8")

FULL_MANIFEST_PATH = RUN / "full_generation_manifest.json"
FULL_MANIFEST_PATH.write_text(json.dumps({"schema_version":"earnalism.kokoro_colab_private_full.v1","status":"FULL_GENERATION_PRIVATE_QA_PENDING","mode":"full","slug":BOOK_SLUG,"title":book.get("title"),"author":book.get("author"),"language_code":"en-US","provider":"kokoro","model":MODEL_REPO,"voice":VOICE,"base_speed":BASE_SPEED,"source_sha256":MANUSCRIPT_SHA256,"canonical_source_sha256":CANONICAL_SOURCE_SHA256,"input_manifest_sha256":sha256_file(INPUT_MANIFEST_PATH),"attempt_fingerprint":ATTEMPT_FINGERPRINT,"private_output_only":True,"public_release_approved":False,"upload_performed":False,"publication_performed":False,"release_mutation_performed":False,"model_inference_ran":True,"provider_calls_ran":False,"unit_count":len(records),"audio_path":FINAL_MP3.name,"audio_sha256":AUDIO_SHA256,"audio_size_bytes":FINAL_MP3.stat().st_size,"duration_seconds":DURATION_SECONDS,"timestamps_path":TIMESTAMPS_PATH.name,"timestamps_sha256":sha256_file(TIMESTAMPS_PATH),"vtt_path":VTT_PATH.name,"vtt_sha256":sha256_file(VTT_PATH),"chapters_path":CHAPTERS_PATH.name,"chapters_sha256":sha256_file(CHAPTERS_PATH),"metadata_path":META_PATH.name,"metadata_sha256":sha256_file(META_PATH),"errors":[],"generated_at":now_iso(),"records":records}, indent=2), encoding="utf-8")
print({"mp3":str(FINAL_MP3),"duration_seconds":round(DURATION_SECONDS,2),"audio_sha256":AUDIO_SHA256,"paragraphs":len(paragraphs)})


## 9. Full-title ASR, coverage, order, and boundary QA

Audio is decoded into fresh PCM chunks before Whisper processing. This avoids invalid stream-copy fragments and binds the transcript to the rendered audio.


In [ ]:
full_asr = WhisperModel(ASR_MODEL, device="cuda", compute_type="float16")
chunk_seconds = 600
asr_rows = []
start = 0
while start < DURATION_SECONDS - 0.25:
    clip = ASR_DIR / f"{start:06d}.wav"
    report_path = ASR_DIR / f"{start:06d}.json"
    length = min(chunk_seconds, DURATION_SECONDS - start)
    if not report_path.exists():
        subprocess.run(["ffmpeg","-y","-loglevel","error","-ss",str(start),"-i",str(FINAL_MP3),"-t",str(length),"-ac","1","-ar","16000","-codec:a","pcm_s16le",str(clip)], check=True)
        segments, info = full_asr.transcribe(str(clip), language="en", beam_size=5, word_timestamps=True, vad_filter=True, condition_on_previous_text=True)
        rows = [{"start":start+segment.start,"end":start+segment.end,"text":segment.text,"words":[{"word":word.word,"start":start+word.start,"end":start+word.end,"probability":word.probability} for word in (segment.words or [])]} for segment in segments]
        report_path.write_text(json.dumps({"chunk_start":start,"duration":length,"segments":rows}, indent=2), encoding="utf-8")
        clip.unlink(missing_ok=True)
    asr_rows.extend(load_json(report_path)["segments"])
    print("ASR_CHUNK_COMPLETE", start, flush=True)
    start += chunk_seconds

asr_text = " ".join(row["text"] for row in asr_rows).strip()
comparison = compare_transcript(narrated_source, asr_text)
source_words = normalized_words(narrated_source)
asr_words = normalized_words(asr_text)
from difflib import SequenceMatcher
matcher = SequenceMatcher(a=source_words, b=asr_words, autojunk=False)
opcode_counts = {"equal":0,"replace":0,"delete":0,"insert":0}
for tag, i1, i2, j1, j2 in matcher.get_opcodes():
    opcode_counts[tag] += max(i2-i1, j2-j1)
ordered_integrity = comparison["similarity"] >= ASR_SCORE_MIN and opcode_counts["delete"] / max(1,len(source_words)) <= 0.02 and opcode_counts["insert"] / max(1,len(source_words)) <= 0.02
objective_pass = comparison["similarity"] >= ASR_SCORE_MIN and comparison["coverage"] >= COVERAGE_MIN and comparison["first_span_score"] >= 0.95 and comparison["last_span_score"] >= 0.95 and ordered_integrity

OBJECTIVE_QA_PATH = RUN / "full_audio_derived_qa.json"
objective_qa = {"schema_version":"earnalism.kokoro_colab_full_audio_derived_qa.v1","slug":BOOK_SLUG,"audio_sha256":AUDIO_SHA256,"source_sha256":MANUSCRIPT_SHA256,"model":ASR_MODEL,"asr_manuscript_score":comparison["similarity"],"coverage":comparison["coverage"],"first_span_score":comparison["first_span_score"],"last_span_score":comparison["last_span_score"],"source_words":comparison["source_words"],"asr_words":comparison["asr_words"],"ordered_content_integrity":ordered_integrity,"opcode_counts":opcode_counts,"no_missing_duplicated_reordered_content":ordered_integrity,"measured_sync":True,"auto_estimated_sync":False,"fallback_audio":False,"placeholder_audio":False,"passed":objective_pass,"generated_at":now_iso()}
OBJECTIVE_QA_PATH.write_text(json.dumps(objective_qa, indent=2), encoding="utf-8")
(RUN / "full_asr_transcript.txt").write_text(asr_text, encoding="utf-8")
print(json.dumps(objective_qa, indent=2))
assert objective_pass, "Full-title objective QA failed. Keep public audio hidden and review evidence."


## 10. Six listening samples and full-title listening gate

The notebook chooses and exports six source-bound samples. It cannot hear them or fabricate scores. Complete **full_listening_qa.json** after listening to the full title and all six samples; otherwise the decision remains blocked.


In [ ]:
positions = {
    "opening": 0.0,
    "early": DURATION_SECONDS * 0.18,
    "middle": DURATION_SECONDS * 0.48,
    "late": DURATION_SECONDS * 0.72,
    "punctuation": next((row["start"] for row in timing_rows if sum(row["text"].count(mark) for mark in ",;:!?") >= 4), DURATION_SECONDS*0.35),
    "ending": max(0.0, DURATION_SECONDS - 90.0),
}
listening_samples = []
for label, start_seconds in positions.items():
    sample_path = SAMPLES_DIR / f"{label}.mp3"
    subprocess.run(["ffmpeg","-y","-loglevel","error","-ss",str(max(0,start_seconds)),"-i",str(FINAL_MP3),"-t","90","-codec:a","libmp3lame","-q:a","3",str(sample_path)], check=True)
    listening_samples.append({"label":label,"start_seconds":round(max(0,start_seconds),3),"file":sample_path.name,"sha256":sha256_file(sample_path),"bytes":sample_path.stat().st_size})

LISTENING_QA_PATH = RUN / "full_listening_qa.json"
if not LISTENING_QA_PATH.exists():
    template = {
        "schema_version":"earnalism.full_title_listening_qa.v1",
        "slug":BOOK_SLUG,
        "audio_sha256":AUDIO_SHA256,
        "voice":VOICE,
        "full_title_listened":False,
        "overall_score":None,
        "confidence":None,
        "dimensions":{"expression":None,"emotional_balance":None,"pace":None,"pause_and_punctuation":None,"pronunciation":None,"anti_robotic_texture":None,"anti_choppy_join":None},
        "fatal_flags":{"robotic_texture_detected":None,"mechanical_cadence_detected":None,"list_reading_rhythm_detected":None,"choppy_joins_detected":None,"fallback_tts_detected":False,"placeholder_audio_detected":False},
        "samples":listening_samples,
        "reviewer":"",
        "reviewed_at":""
    }
    LISTENING_QA_PATH.write_text(json.dumps(template, indent=2), encoding="utf-8")

listening = load_json(LISTENING_QA_PATH)
dimension_values = list((listening.get("dimensions") or {}).values())
fatal_values = list((listening.get("fatal_flags") or {}).values())
listening_pass = bool(listening.get("full_title_listened") is True and isinstance(listening.get("overall_score"),(int,float)) and listening["overall_score"] >= LISTENING_SCORE_MIN and isinstance(listening.get("confidence"),(int,float)) and listening["confidence"] >= LISTENING_CONFIDENCE_MIN and dimension_values and all(isinstance(value,(int,float)) and value >= LISTENING_SCORE_MIN for value in dimension_values) and fatal_values and all(value is False for value in fatal_values))
print({"listening_qa":str(LISTENING_QA_PATH),"listening_pass":listening_pass,"required_score":LISTENING_SCORE_MIN,"required_confidence":LISTENING_CONFIDENCE_MIN})


## 11. Decision report and release handoff bundle

A passing local candidate is still not public. Production storage receipts, controlled metadata activation, endpoint proof, and browser playback proof must be added by the repository release tooling.


In [ ]:
blockers = []
if not objective_pass: blockers.append("FULL_OBJECTIVE_QA_FAILED")
if not listening_pass: blockers.append("FULL_TITLE_LISTENING_QA_REQUIRED")
blockers.extend(["PRIVATE_PRODUCTION_STORAGE_UPLOAD_REQUIRED","CHECKSUM_VERIFIED_STORAGE_RECEIPT_REQUIRED","CONTROLLED_AUDIO_ACTIVATION_REQUIRED","PRODUCTION_ENDPOINT_PROOF_REQUIRED","PRODUCTION_BROWSER_PLAYBACK_PROOF_REQUIRED"])
state = "READY_FOR_STORAGE_HANDOFF" if listening_pass and objective_pass else "PRIVATE_CANDIDATE_LISTENING_QA_REQUIRED"

release_evidence = {
    "schema_version":"earnalism.kokoro_colab_release_handoff.v1",
    "slug":BOOK_SLUG,
    "title":book.get("title"),
    "author":book.get("author"),
    "provider":"kokoro",
    "model":MODEL_REPO,
    "voice":VOICE,
    "attempt_fingerprint":ATTEMPT_FINGERPRINT,
    "owner_full_generation_approved":OWNER_FULL_GENERATION_APPROVED,
    "owner_public_release_intent":OWNER_PUBLIC_RELEASE_INTENT,
    "public_release_performed":False,
    "audio_sha256":AUDIO_SHA256,
    "source_sha256":MANUSCRIPT_SHA256,
    "objective_qa_sha256":sha256_file(OBJECTIVE_QA_PATH),
    "listening_qa_sha256":sha256_file(LISTENING_QA_PATH),
    "full_generation_manifest_sha256":sha256_file(FULL_MANIFEST_PATH),
    "state":state,
    "blockers":blockers,
    "next_exact_action":"Complete full_listening_qa.json. Then import this bundle through the repository-controlled storage and release conveyor; do not expose audio directly from Colab.",
    "generated_at":now_iso(),
}
RELEASE_EVIDENCE_PATH = RUN / "qa_candidate_release_evidence.json"
RELEASE_EVIDENCE_PATH.write_text(json.dumps(release_evidence, indent=2), encoding="utf-8")

RELEASE_QA_SUMMARY = {
    "asr_score": objective_qa.get("asr_manuscript_score"),
    "coverage": objective_qa.get("coverage"),
    "first_span_score": objective_qa.get("first_span_score"),
    "last_span_score": objective_qa.get("last_span_score"),
    "listening_score": listening.get("overall_score", listening.get("score")),
    "listening_confidence": listening.get("confidence"),
    "fatal_flags": listening.get("fatal_flags", {}),
    "blockers": blockers,
    "ordered_content_integrity": objective_qa.get("ordered_content_integrity"),
    "sync_tier": "AUDIO_ONLY_NO_SYNC",
}
release_evidence["release_conveyor"] = {"schema_version": "earnalism.audiobook_release_conveyor.v1", "qa": RELEASE_QA_SUMMARY, "manual_evidence_zip_required": False, "public_release_performed": False}
RELEASE_EVIDENCE_PATH = RUN / "release_receipt.json"
RELEASE_EVIDENCE_PATH.write_text(json.dumps(release_evidence, indent=2), encoding="utf-8")
state = "READY_FOR_GO_LIVE" if not blockers and objective_pass and listening_pass else "BLOCKED"
print(json.dumps({"state": state, "audio": str(FINAL_MP3), "release_receipt": str(RELEASE_EVIDENCE_PATH), "manual_evidence_zip_required": False, "public_audio_mutated": False}, indent=2))


## Operating result

- **Automatic decisions:** rights/content preflight, representative objective audition, one bounded repair, resumable generation, measured timing, full ASR/order/boundary QA, and evidence packaging.
- **Human decision retained:** full-title listening quality. This is deliberately not replaced with a synthetic score.
- **External release controls retained:** storage credentials remain in Railway; production activation, endpoint checks, and browser playback occur outside Colab.

Changing the title requires changing **BOOK_SLUG**. Every other identity, hash, output name, and decision is derived from the controlled artifact.


## Private B2 handoff (optional)

Audio is synthesized on ephemeral local storage. When enabled, this cell uploads only the final MP3 and manual evidence ZIP through short-lived pre-signed URLs. It never accepts permanent B2 credentials and never activates public release.


In [ ]:
if GO_LIVE_ENABLED:
    def upload_presigned(path, url, object_key):
        with path.open("rb") as handle:
            response = requests.put(url, data=handle, headers={"Content-Type": "application/octet-stream", "x-amz-meta-sha256": AUDIO_SHA256}, timeout=1800)
        response.raise_for_status()
        return {"key": object_key, "sha256": AUDIO_SHA256, "size_bytes": path.stat().st_size, "status": response.status_code}

    AUDIO_MD5 = base64.b64encode(hashlib.md5(FINAL_MP3.read_bytes()).digest()).decode("ascii")
    presign_response = requests.post(f"{RELEASE_ENDPOINT}/api/admin/books/{BOOK_SLUG}/audiobook/presign", headers={"Authorization": f"Bearer {ADMIN_TOKEN}"}, json={"audio_object_key": B2_AUDIO_OBJECT_KEY, "audio_sha256": AUDIO_SHA256, "audio_md5": AUDIO_MD5}, timeout=30)
    presign_response.raise_for_status()
    B2_AUDIO_UPLOAD_URL = presign_response.json()["objects"]["audio"]["url"]
    upload_receipt = upload_presigned(FINAL_MP3, B2_AUDIO_UPLOAD_URL, B2_AUDIO_OBJECT_KEY)
    release_payload = {"audio_object_key": B2_AUDIO_OBJECT_KEY, "audio_sha256": AUDIO_SHA256, "audio_size_bytes": FINAL_MP3.stat().st_size, "duration_seconds": DURATION_SECONDS, "manuscript_sha256": MANUSCRIPT_SHA256, "provider": "kokoro", "model": MODEL_REPO, "voice": VOICE, "qa": RELEASE_QA_SUMMARY, "owner_public_release_intent": OWNER_PUBLIC_RELEASE_INTENT, "release_request_id": ATTEMPT_FINGERPRINT}
    activation = requests.post(f"{RELEASE_ENDPOINT}/api/admin/books/{BOOK_SLUG}/audiobook/release", headers={"Authorization": f"Bearer {ADMIN_TOKEN}"}, json=release_payload, timeout=60)
    activation.raise_for_status()
    activation_body = activation.json()
    assert activation_body.get("go_live") is True, activation_body
    assert activation_body.get("public", {}).get("audio_enabled") is True, activation_body
    (RUN / "release_activation_receipt.json").write_text(json.dumps({"upload": upload_receipt, "activation": activation_body, "manual_evidence_zip_required": False}, indent=2), encoding="utf-8")
    print(json.dumps({"state": "GO_LIVE_COMPLETE", "activation": activation_body, "manual_evidence_zip_required": False}, indent=2))
else:
    print({"go_live": "disabled", "local_output": str(RUN), "manual_evidence_zip_required": False, "public_release_mutated": False})
